# Notebook 10 — Structure from Motion

**Vision & 3D Mapping Workshop** | Block 3: Depth & 3D Reconstruction

---

## Why This Matters

Structure from Motion (SfM) is the algorithmic backbone of modern 3-D reconstruction.
Given a collection of **unordered images** — from a phone camera walk-around, a drone
survey, or internet photo collections — SfM simultaneously recovers:

1. The **3-D structure** of the scene (a sparse point cloud)
2. The **camera pose** (position and orientation) for every image

SfM is the engine behind COLMAP, Agisoft Metashape, and the initialization stage of
Neural Radiance Fields (NeRF) and 3D Gaussian Splatting.

This notebook derives every component from first principles — PnP, triangulation,
bundle adjustment with the Schur complement — and implements a mini SfM pipeline
on synthetic data.

### What You'll Learn

1. **SfM overview** — unordered images → 3D reconstruction + camera poses
2. **Incremental SfM pipeline** — initialize → register → triangulate → BA
3. **PnP** — solving for camera pose from 3D-2D correspondences
4. **Bundle Adjustment** — full derivation, Jacobian structure, Schur complement
5. **Gauss-Newton / Levenberg-Marquardt** — optimization theory
6. **Mini SfM** — 5-view synthetic reconstruction from scratch
7. **COLMAP and MVS** — overview of production systems

### Prerequisites

| Concept | Where |
|:---|:---|
| Pinhole camera model, $K$, projection | Notebook 03 |
| Rotations, $\text{SO}(3)$, axis-angle | Notebook 04 |
| Feature matching, RANSAC | Notebook 05 |
| Fundamental / Essential matrix | Notebook 05 |

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "image.cmap": "gray",
    "axes.grid": False,
})

---
## 1. SfM Overview

### 1.1 Problem Statement

**Given**: A set of images $\{I_1, I_2, \ldots, I_n\}$ of the same scene, possibly
unordered and with unknown camera poses.

**Recover**:
- Camera poses $\{(R_i, \mathbf{t}_i)\}_{i=1}^n$ for each image
- 3-D point cloud $\{\mathbf{X}_j\}_{j=1}^M$

such that the projections of the 3-D points match the observed 2-D keypoints:

$$
\mathbf{x}_{ij} \approx \pi\bigl(K_i [R_i \mid \mathbf{t}_i] \, \mathbf{X}_j\bigr)
$$

### 1.2 Two Flavours of SfM

| Approach | Strategy | Pros | Cons |
|:---|:---|:---|:---|
| **Incremental** | Add one image at a time, refine with BA | Robust, handles outliers | Slow for many images, drift |
| **Global** | Estimate all rotations first, then translations | Fast, no drift | Sensitive to outliers |

This notebook focuses on **incremental SfM** (the COLMAP approach).

### 1.3 The Scale Ambiguity

From images alone, SfM can only recover the scene **up to a global scale**:
if $(\{R_i, \mathbf{t}_i\}, \{\mathbf{X}_j\})$ is a solution, then so is
$(\{R_i, s \cdot \mathbf{t}_i\}, \{s \cdot \mathbf{X}_j\})$ for any $s > 0$.

To obtain metric scale, you need external information (known object size,
GPS, IMU, etc.).

---
## 2. Incremental SfM Pipeline

### 2.1 Algorithm

```
1. DETECT features and MATCH across all image pairs
2. SELECT best initialization pair (most inliers, wide baseline)
3. ESTIMATE F → E → decompose → (R, t) for the initial pair
4. TRIANGULATE initial 3-D points
5. REPEAT for each remaining image:
   a. Find 2D↔3D correspondences (PnP candidates)
   b. REGISTER via PnP-RANSAC → camera pose (R_i, t_i)
   c. TRIANGULATE new 3-D points
   d. BUNDLE ADJUST all cameras and points
6. RETURN cameras + point cloud
```

### 2.2 Initialization

The initial pair must have:
- Many feature matches (geometric consistency)
- Sufficient baseline (not pure rotation, which gives degenerate triangulation)

Steps:
1. Estimate fundamental matrix $F$ with RANSAC
2. Compute essential matrix: $E = K^T F K$
3. Decompose $E$ into four $(R, \mathbf{t})$ candidates
4. Select the solution where all points are in front of both cameras
   (**cheirality check**)
5. Triangulate the initial 3-D points

In [ ]:
def visualize_sfm_pipeline():
    """Create a visual diagram of the incremental SfM pipeline."""
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 8)
    ax.axis('off')
    
    steps = [
        (1.5, 7.0, 2.0, 1.0, 'Feature Detection\n& Matching', '#FFB3BA'),
        (1.5, 5.5, 2.0, 1.0, 'Select Best Pair\n(most inliers)', '#BAFFC9'),
        (1.5, 4.0, 2.0, 1.0, 'Estimate E\n→ Decompose (R,t)', '#BAE1FF'),
        (1.5, 2.5, 2.0, 1.0, 'Triangulate\nInitial Points', '#FFFFBA'),
    ]
    
    loop_steps = [
        (6.0, 6.25, 2.5, 1.0, 'PnP-RANSAC\nRegister New Image', '#E8BAFF'),
        (10.0, 6.25, 2.5, 1.0, 'Triangulate\nNew Points', '#FFDFBA'),
        (10.0, 4.25, 2.5, 1.0, 'Bundle\nAdjustment', '#C9FFE5'),
        (6.0, 4.25, 2.5, 1.0, 'More images?', '#FFE4E1'),
    ]
    
    for x, y, w, h, text, color in steps:
        rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='black',
                             facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=10, fontweight='bold', zorder=3)
    
    for i in range(len(steps) - 1):
        ax.annotate('', xy=(2.5, steps[i+1][1] + steps[i+1][3]),
                     xytext=(2.5, steps[i][1]),
                     arrowprops=dict(arrowstyle='->', lw=2))
    
    for x, y, w, h, text, color in loop_steps:
        rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='black',
                             facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=10, fontweight='bold', zorder=3)
    
    ax.annotate('', xy=(6.0, 6.75), xytext=(3.5, 3.0),
                arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
    ax.annotate('', xy=(10.0, 6.75), xytext=(8.5, 6.75),
                arrowprops=dict(arrowstyle='->', lw=2))
    ax.annotate('', xy=(11.25, 5.25), xytext=(11.25, 6.25),
                arrowprops=dict(arrowstyle='->', lw=2))
    ax.annotate('', xy=(8.5, 4.75), xytext=(10.0, 4.75),
                arrowprops=dict(arrowstyle='->', lw=2))
    ax.annotate('', xy=(7.25, 5.25), xytext=(7.25, 4.25 + 1.0),
                arrowprops=dict(arrowstyle='->', lw=2, color='green'))
    ax.text(7.25, 5.6, 'Yes', fontsize=10, ha='center', color='green')
    
    ax.annotate('', xy=(7.25, 2.5), xytext=(7.25, 4.25),
                arrowprops=dict(arrowstyle='->', lw=2, color='red'))
    ax.text(7.6, 3.4, 'No', fontsize=10, color='red')
    
    rect_final = plt.Rectangle((5.5, 1.5, ), 3.5, 1.0, linewidth=2,
                               edgecolor='black', facecolor='gold', zorder=2)
    ax.add_patch(rect_final)
    ax.text(7.25, 2.0, 'Final Reconstruction\nCameras + Point Cloud',
            ha='center', va='center', fontsize=11, fontweight='bold', zorder=3)
    
    ax.text(8, 7.5, 'Incremental SfM Pipeline', fontsize=16,
            fontweight='bold', ha='center')
    plt.tight_layout()
    plt.show()

visualize_sfm_pipeline()

---
## 3. Perspective-n-Point (PnP)

### 3.1 Problem Statement

Given $n$ correspondences between **known** 3-D points $\{\mathbf{X}_j\}$ and
their 2-D projections $\{\mathbf{x}_j\}$ in an image, find the camera pose
$(R, \mathbf{t})$:

$$
\min_{R \in \text{SO}(3), \, \mathbf{t} \in \mathbb{R}^3}
\sum_{j=1}^{n}
\bigl\|\pi\bigl(K [R \mid \mathbf{t}] \, \tilde{\mathbf{X}}_j\bigr) - \mathbf{x}_j\bigr\|^2
$$

where $\tilde{\mathbf{X}}_j = [X_j, Y_j, Z_j, 1]^T$ is the homogeneous 3-D point.

### 3.2 The Projection Function

The projection of a 3-D world point to a 2-D pixel:

$$
\pi(K [R \mid \mathbf{t}] \, \tilde{\mathbf{X}}) = \pi\left(K \begin{bmatrix} R \mathbf{X} + \mathbf{t} \end{bmatrix}\right)
$$

Let $\mathbf{X}_c = R \mathbf{X} + \mathbf{t}$ be the point in camera coordinates:

$$
\mathbf{x} = \begin{bmatrix} f_x \dfrac{X_c}{Z_c} + c_x \\[6pt] f_y \dfrac{Y_c}{Z_c} + c_y \end{bmatrix}
$$

### 3.3 P3P: The Minimal Solver

**P3P** uses exactly 3 correspondences to solve for camera pose. Given three
3-D points $A, B, C$ and their projections $a, b, c$, we know:

- The angles between viewing rays: $\cos(\angle AOB) = \hat{\mathbf{a}} \cdot \hat{\mathbf{b}}$
- The inter-point distances: $|AB|, |BC|, |AC|$

By the **law of cosines**, for the triangle $OAB$ with $|OA| = s_1, |OB| = s_2$:

$$
|AB|^2 = s_1^2 + s_2^2 - 2 s_1 s_2 \cos(\angle AOB)
$$

This gives a system of 3 equations in 3 unknowns ($s_1, s_2, s_3$), which
reduces to a degree-4 polynomial → up to **4 solutions**.

A 4th point disambiguates. In RANSAC, we use the 4th point for verification.

### 3.4 PnP with RANSAC

For $n > 3$ correspondences with outliers:

1. Sample 4 correspondences randomly
2. Solve P3P + use 4th point for disambiguation
3. Count inliers (reprojection error < threshold)
4. Repeat; keep the best model
5. Refine on all inliers using iterative PnP (Levenberg-Marquardt)

In [ ]:
def project_points(X_world, R, t, K):
    """Project 3D world points to 2D image coordinates."""
    X_cam = (R @ X_world.T).T + t.reshape(1, 3)
    x_proj = np.zeros((len(X_world), 2))
    valid = X_cam[:, 2] > 0
    x_proj[valid, 0] = K[0, 0] * X_cam[valid, 0] / X_cam[valid, 2] + K[0, 2]
    x_proj[valid, 1] = K[1, 1] * X_cam[valid, 1] / X_cam[valid, 2] + K[1, 2]
    return x_proj, valid


def solve_pnp(pts_3d, pts_2d, K):
    """Solve PnP using OpenCV (wraps EPnP + RANSAC)."""
    ok, rvec, tvec, inliers = cv2.solvePnPRansac(
        pts_3d.astype(np.float64),
        pts_2d.astype(np.float64),
        K.astype(np.float64),
        distCoeffs=None,
        reprojectionError=3.0,
        iterationsCount=1000,
        flags=cv2.SOLVEPNP_ITERATIVE,
    )
    if not ok:
        return None, None, None
    R, _ = cv2.Rodrigues(rvec)
    return R, tvec.ravel(), inliers


K_sfm = np.array([[500, 0, 320],
                   [0, 500, 240],
                   [0, 0, 1]], dtype=np.float64)

rng = np.random.RandomState(42)
pts_3d_test = rng.randn(50, 3) * 2 + np.array([0, 0, 8])

R_true = Rotation.from_euler('xyz', [10, -15, 5], degrees=True).as_matrix()
t_true = np.array([0.5, -0.3, 0.1])

pts_2d_test, valid = project_points(pts_3d_test, R_true, t_true, K_sfm)

noise = rng.randn(len(pts_2d_test), 2) * 1.0
pts_2d_noisy = pts_2d_test + noise

R_est, t_est, inliers = solve_pnp(pts_3d_test[valid], pts_2d_noisy[valid], K_sfm)

if R_est is not None:
    print("=== PnP Results ===")
    print(f"\nTrue R:\n{R_true}")
    print(f"\nEstimated R:\n{R_est}")
    print(f"\nTrue t: {t_true}")
    print(f"Est  t: {t_est}")
    
    R_err = np.arccos(np.clip((np.trace(R_est @ R_true.T) - 1) / 2, -1, 1))
    t_err = np.linalg.norm(t_est - t_true)
    print(f"\nRotation error: {np.degrees(R_err):.4f}°")
    print(f"Translation error: {t_err:.6f} m")
    print(f"Inliers: {len(inliers)}/{valid.sum()}")

---
## 4. Bundle Adjustment — Full Derivation

### 4.1 The Optimization Problem

Bundle adjustment jointly refines **all** camera parameters and **all** 3-D points
by minimising the total reprojection error:

$$
\min_{\{R_i, \mathbf{t}_i\}, \{\mathbf{X}_j\}}
\sum_{i=1}^{n} \sum_{j \in \mathcal{V}(i)}
\bigl\| \pi\bigl(K_i [R_i \mid \mathbf{t}_i] \, \mathbf{X}_j\bigr) - \mathbf{x}_{ij} \bigr\|^2
$$

where $\mathcal{V}(i)$ is the set of 3-D points visible in image $i$, and
$\mathbf{x}_{ij}$ is the observed 2-D projection.

### 4.2 Parameterisation

**Camera $i$**: 6 parameters — axis-angle rotation $\boldsymbol{\rho}_i \in \mathbb{R}^3$
(via the exponential map $R_i = \exp([\boldsymbol{\rho}_i]_\times)$) and
translation $\mathbf{t}_i \in \mathbb{R}^3$.

**Point $j$**: 3 parameters — $\mathbf{X}_j = (X_j, Y_j, Z_j)^T$.

Total parameters: $\theta = [\underbrace{\boldsymbol{\rho}_1, \mathbf{t}_1, \ldots, \boldsymbol{\rho}_n, \mathbf{t}_n}_{6n \text{ camera params}}, \underbrace{\mathbf{X}_1, \ldots, \mathbf{X}_M}_{3M \text{ point params}}]$

### 4.3 The Residual Vector

For observation $(i, j)$, the residual is the 2-D reprojection error:

$$
\mathbf{r}_{ij}(\theta) = \pi\bigl(K_i [R_i \mid \mathbf{t}_i] \, \mathbf{X}_j\bigr) - \mathbf{x}_{ij} \in \mathbb{R}^2
$$

Stack all residuals:

$$
\mathbf{r}(\theta) = \begin{bmatrix} \mathbf{r}_{i_1 j_1} \\ \mathbf{r}_{i_2 j_2} \\ \vdots \end{bmatrix} \in \mathbb{R}^{2K}
$$

where $K$ is the total number of observations.

### 4.4 Jacobian Structure

The Jacobian $J \in \mathbb{R}^{2K \times (6n + 3M)}$ has a very special
**sparse** structure. Each residual $\mathbf{r}_{ij}$ depends on only:

- Camera $i$'s 6 parameters
- Point $j$'s 3 parameters

So each $2 \times (6n + 3M)$ row block of $J$ has only **9 non-zero columns** (18 scalar entries):

$$
J_{ij} = \begin{bmatrix}
\underbrace{0 \cdots 0}_{6(i-1)} & \underbrace{A_{ij}}_{6} & \underbrace{0 \cdots 0}_{6(n-i)} &
\underbrace{0 \cdots 0}_{3(j-1)} & \underbrace{B_{ij}}_{3} & \underbrace{0 \cdots 0}_{3(M-j)}
\end{bmatrix}
$$

where:
- $A_{ij} = \frac{\partial \mathbf{r}_{ij}}{\partial \text{cam}_i} \in \mathbb{R}^{2 \times 6}$
- $B_{ij} = \frac{\partial \mathbf{r}_{ij}}{\partial \mathbf{X}_j} \in \mathbb{R}^{2 \times 3}$

#### The Visibility Matrix

The **visibility matrix** $W \in \{0,1\}^{n \times M}$ encodes which points are observed
by which cameras: $W_{ij} = 1$ if point $j$ appears in image $i$ (i.e. $j \in \mathcal{V}(i)$).
Equivalently, $\mathcal{V}(i) = \{j : W_{ij} = 1\}$ and $\mathcal{C}(j) = \{i : W_{ij} = 1\}$.

$W$ fully determines the **sparsity pattern** of the Jacobian $J$: the row block for
observation $(i, j)$ has non-zero entries only in the columns for camera $i$ and
point $j$. In practice $W$ is very sparse (each image sees $\ll M$ points), which is
why BA can scale to millions of points.

#### Deriving $A_{ij}$ and $B_{ij}$: Chain Rule Through Projection

Each residual decomposes as a chain of three maps:

$$
\text{cam params} \;\xrightarrow{\text{transform}}\; \mathbf{X}_c \;\xrightarrow{\text{project}}\; \mathbf{x}_n \;\xrightarrow{\text{calibrate}}\; \mathbf{x}_p
$$

Let $\mathbf{X}_c = R\mathbf{X} + \mathbf{t}$ (camera-frame point), $\mathbf{x}_n = [X_c/Z_c,\; Y_c/Z_c]^\top$
(normalised image point), $\mathbf{x}_p = [f_x x_n + c_x,\; f_y y_n + c_y]^\top$ (pixel).

**Step 1 — Projection Jacobian** $J_{\pi} \in \mathbb{R}^{2 \times 3}$:

$$
\frac{\partial \mathbf{x}_n}{\partial \mathbf{X}_c}
= \frac{1}{Z_c}\begin{bmatrix} 1 & 0 & -X_c/Z_c \\ 0 & 1 & -Y_c/Z_c \end{bmatrix}
$$

Combined with the calibration matrix:

$$
J_\pi = \frac{\partial \mathbf{x}_p}{\partial \mathbf{X}_c}
= \frac{1}{Z_c}\begin{bmatrix} f_x & 0 & -f_x X_c/Z_c \\ 0 & f_y & -f_y Y_c/Z_c \end{bmatrix}
$$

**Step 2 — Jacobian w.r.t. point** $B_{ij} \in \mathbb{R}^{2 \times 3}$:

Since $\mathbf{X}_c = R_i \mathbf{X}_j + \mathbf{t}_i$, we have
$\partial \mathbf{X}_c / \partial \mathbf{X}_j = R_i$, so:

$$
\boxed{B_{ij} = J_\pi \, R_i}
$$

**Step 3 — Jacobian w.r.t. camera** $A_{ij} \in \mathbb{R}^{2 \times 6}$:

Parameterise the camera pose via the Lie algebra: perturb $R_i \leftarrow R_i \,\text{Exp}(\delta\boldsymbol{\rho})$,
$\mathbf{t}_i \leftarrow \mathbf{t}_i + \delta\mathbf{t}$. The perturbed camera-frame point is:

$$
\mathbf{X}_c' = R_i(I + [\delta\boldsymbol{\rho}]_\times)\mathbf{X}_j + \mathbf{t}_i + \delta\mathbf{t}
= \mathbf{X}_c + R_i [\delta\boldsymbol{\rho}]_\times \mathbf{X}_j + \delta\mathbf{t}
$$

Using $[\delta\boldsymbol{\rho}]_\times \mathbf{X}_j = -[\mathbf{X}_j]_\times \delta\boldsymbol{\rho}$:

$$
\frac{\partial \mathbf{X}_c}{\partial [\delta\boldsymbol{\rho},\; \delta\mathbf{t}]}
= \bigl[\;-R_i[\mathbf{X}_j]_\times \;\big|\; I_3\;\bigr] \in \mathbb{R}^{3 \times 6}
$$

Applying the chain rule:

$$
\boxed{
A_{ij} = J_\pi \bigl[\;-R_i [\mathbf{X}_j]_\times \;\big|\; I_3\;\bigr]
= \frac{1}{Z_c}\begin{bmatrix} f_x & 0 & -f_x X_c/Z_c \\ 0 & f_y & -f_y Y_c/Z_c \end{bmatrix}
\bigl[\;-R_i [\mathbf{X}_j]_\times \;\big|\; I_3\;\bigr]
}
$$

In summary, the chain rule gives:

$$
\frac{d}{d\boldsymbol{\xi}} \pi\bigl(\text{Exp}(\boldsymbol{\xi}) \cdot \mathbf{X}\bigr)
= \underbrace{J_\pi}_{\text{projection}} \cdot \underbrace{J_{\text{pose}}}_{\text{Lie perturbation}}
$$

This decomposition is implemented once and reused for every observation — the
key to efficient analytic Jacobians in BA.

### 4.5 The Normal Equations

The Gauss-Newton update solves:

$$
J^T J \, \delta = -J^T \mathbf{r}
$$

The **Hessian approximation** $H = J^T J$ inherits the block structure of $J$.
Partitioning into camera ($c$) and point ($p$) blocks:

$$
H = \begin{bmatrix} H_{cc} & H_{cp} \\ H_{pc} & H_{pp} \end{bmatrix}, \qquad
J^T \mathbf{r} = \begin{bmatrix} \mathbf{g}_c \\ \mathbf{g}_p \end{bmatrix}
$$

where:
- $H_{cc} \in \mathbb{R}^{6n \times 6n}$: camera-camera block (block diagonal with $6 \times 6$ blocks!)
- $H_{pp} \in \mathbb{R}^{3M \times 3M}$: point-point block (**block diagonal** with $3 \times 3$ blocks!)
- $H_{cp} \in \mathbb{R}^{6n \times 3M}$: camera-point cross terms (sparse)

### 4.6 Why $H_{pp}$ Is Block-Diagonal

Each 3-D point $j$ only contributes to residuals involving itself. Therefore:

$$
(H_{pp})_{jj} = \sum_{i \in \mathcal{C}(j)} B_{ij}^T B_{ij} \in \mathbb{R}^{3 \times 3}
$$

and $(H_{pp})_{jk} = 0$ for $j \neq k$. This makes $H_{pp}$ trivially invertible
as $M$ independent $3 \times 3$ inversions.

Similarly, $H_{cc}$ is block-diagonal because each camera's parameters interact
with the points it observes, not with other cameras directly:

$$
(H_{cc})_{ii} = \sum_{j \in \mathcal{V}(i)} A_{ij}^T A_{ij} \in \mathbb{R}^{6 \times 6}
$$

### 4.7 The Schur Complement

The normal equations:

$$
\begin{bmatrix} H_{cc} & H_{cp} \\ H_{pc} & H_{pp} \end{bmatrix}
\begin{bmatrix} \delta_c \\ \delta_p \end{bmatrix}
= -\begin{bmatrix} \mathbf{g}_c \\ \mathbf{g}_p \end{bmatrix}
$$

From the second row: $H_{pc} \delta_c + H_{pp} \delta_p = -\mathbf{g}_p$

Solve for $\delta_p$:

$$
\delta_p = H_{pp}^{-1} (-\mathbf{g}_p - H_{pc} \delta_c)
$$

Substitute into the first row:

$$
\bigl(H_{cc} - H_{cp} H_{pp}^{-1} H_{pc}\bigr) \delta_c = -(\mathbf{g}_c - H_{cp} H_{pp}^{-1} \mathbf{g}_p)
$$

Define the **Schur complement**:

$$
\boxed{S = H_{cc} - H_{cp} \, H_{pp}^{-1} \, H_{pc}}
$$

**Key insight**: $S \in \mathbb{R}^{6n \times 6n}$ is **much smaller** than the
full Hessian when there are many points ($M \gg n$). Typical SfM has:

- $n \sim 100$ cameras → $S$ is $600 \times 600$
- $M \sim 100{,}000$ points → full $H$ would be $300{,}600 \times 300{,}600$

This computational efficiency is what makes bundle adjustment feasible for real-time SLAM on drones, where the Schur complement reduces the system from millions of point parameters to hundreds of camera parameters.

**Algorithm**:
1. Form $S$ and the reduced RHS (cheap because $H_{pp}^{-1}$ is block-diagonal)
2. Solve $S \, \delta_c = \mathbf{b}_c$ (small dense system)
3. Back-substitute: $\delta_p = H_{pp}^{-1}(-\mathbf{g}_p - H_{pc} \delta_c)$

This reduces the complexity from $O((6n + 3M)^3)$ to $O((6n)^3 + M)$.

In [ ]:
def visualize_jacobian_sparsity(n_cameras=5, n_points=20, n_obs_per_point=3):
    """Visualize the sparsity pattern of the BA Jacobian and Hessian."""
    rng = np.random.RandomState(42)
    n_obs = n_points * n_obs_per_point
    n_cam_params = 6 * n_cameras
    n_pt_params = 3 * n_points
    total_params = n_cam_params + n_pt_params
    n_residuals = 2 * n_obs
    
    J = np.zeros((n_residuals, total_params))
    row = 0
    for j in range(n_points):
        cams = rng.choice(n_cameras, n_obs_per_point, replace=False)
        for ci in cams:
            J[row:row+2, 6*ci:6*ci+6] = rng.randn(2, 6)
            J[row:row+2, n_cam_params + 3*j:n_cam_params + 3*j+3] = rng.randn(2, 3)
            row += 2
    
    H = J.T @ J
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].spy(J, markersize=1, color='blue')
    axes[0].set_title(f'Jacobian J ({n_residuals}×{total_params})', fontsize=12)
    axes[0].set_xlabel('Parameters [cameras | points]')
    axes[0].set_ylabel('Residuals')
    axes[0].axvline(x=n_cam_params - 0.5, color='red', linestyle='--', alpha=0.7)
    
    axes[1].spy(H, markersize=1, color='blue')
    axes[1].set_title(f'Hessian H = JᵀJ ({total_params}×{total_params})', fontsize=12)
    axes[1].axvline(x=n_cam_params - 0.5, color='red', linestyle='--', alpha=0.7)
    axes[1].axhline(y=n_cam_params - 0.5, color='red', linestyle='--', alpha=0.7)
    axes[1].text(n_cam_params/2, -3, '$H_{cc}$', ha='center', fontsize=11, color='red')
    axes[1].text(n_cam_params + n_pt_params/2, -3, '$H_{pp}$', ha='center',
                 fontsize=11, color='red')
    
    H_pp = H[n_cam_params:, n_cam_params:]
    axes[2].spy(H_pp, markersize=2, color='green')
    axes[2].set_title(f'$H_{{pp}}$ (block-diagonal, {n_pt_params}×{n_pt_params})', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Jacobian J:     {n_residuals} × {total_params}")
    print(f"  Non-zeros:    {np.count_nonzero(J)} / {J.size} ({100*np.count_nonzero(J)/J.size:.1f}%)")
    print(f"Hessian H:      {total_params} × {total_params}")
    print(f"  Non-zeros:    {np.count_nonzero(H)} / {H.size} ({100*np.count_nonzero(H)/H.size:.1f}%)")
    print(f"\nSchur complement S:  {n_cam_params} × {n_cam_params}")
    print(f"Full system:         {total_params} × {total_params}")
    print(f"Reduction ratio:     {total_params**2 / n_cam_params**2:.1f}×")

visualize_jacobian_sparsity()

In [ ]:
def schur_complement_solve(H_cc, H_cp, H_pp, g_c, g_p):
    """
    Solve the normal equations using the Schur complement.
    
    [H_cc  H_cp] [δ_c]   [g_c]
    [H_pc  H_pp] [δ_p] = [g_p]
    
    S = H_cc - H_cp · H_pp⁻¹ · H_pc
    Solve S · δ_c = -(g_c - H_cp · H_pp⁻¹ · g_p)
    Back-substitute: δ_p = H_pp⁻¹ · (-g_p - H_pc · δ_c)
    """
    H_pp_inv = np.linalg.inv(H_pp)
    H_pc = H_cp.T
    
    S = H_cc - H_cp @ H_pp_inv @ H_pc
    
    rhs_c = -(g_c - H_cp @ H_pp_inv @ g_p)
    
    delta_c = np.linalg.solve(S, rhs_c)
    
    delta_p = H_pp_inv @ (-g_p - H_pc @ delta_c)
    
    return delta_c, delta_p, S


n_c, n_p = 12, 9  # camera params, point params
rng = np.random.RandomState(42)
A = rng.randn(30, n_c + n_p)  # synthetic Jacobian
H = A.T @ A + 0.01 * np.eye(n_c + n_p)  # ensure positive definite
g = rng.randn(n_c + n_p)

H_cc = H[:n_c, :n_c]
H_cp = H[:n_c, n_c:]
H_pp = H[n_c:, n_c:]
g_c = g[:n_c]
g_p = g[n_c:]

delta_c_schur, delta_p_schur, S = schur_complement_solve(H_cc, H_cp, H_pp, g_c, g_p)
delta_schur = np.concatenate([delta_c_schur, delta_p_schur])

delta_direct = np.linalg.solve(H, -g)

error = np.linalg.norm(delta_schur - delta_direct)
print(f"Schur complement vs direct solve difference: {error:.2e}")
print(f"Schur complement S size: {S.shape}")
print(f"Full system size: {H.shape}")

---
## 5. Gauss-Newton and Levenberg-Marquardt

### 5.1 Gauss-Newton Method

For the non-linear least-squares problem $\min_\theta \|\mathbf{r}(\theta)\|^2$,
linearise the residual at the current estimate:

$$
\mathbf{r}(\theta + \delta) \approx \mathbf{r}(\theta) + J \delta
$$

Substituting into the cost function:

$$
\|\mathbf{r} + J\delta\|^2 = \mathbf{r}^T\mathbf{r} + 2\mathbf{r}^T J \delta + \delta^T J^T J \delta
$$

Taking the derivative with respect to $\delta$ and setting to zero:

$$
J^T J \, \delta = -J^T \mathbf{r}
$$

This is the **Gauss-Newton update**. It approximates the Hessian as $H \approx J^T J$
(ignoring second-order residual terms).

### 5.2 Levenberg-Marquardt (LM)

LM adds a damping term that interpolates between Gauss-Newton and gradient descent:

$$
\boxed{(J^T J + \lambda I) \, \delta = -J^T \mathbf{r}}
$$

**Behaviour**:

| $\lambda$ | Behaviour | When |
|:---:|:---|:---|
| $\lambda \to 0$ | Pure Gauss-Newton | Near the minimum |
| $\lambda \to \infty$ | Gradient descent with step $\sim 1/\lambda$ | Far from minimum |

**Strategy**: Start with moderate $\lambda$. If a step reduces the cost, accept it
and **decrease** $\lambda$ (trust Gauss-Newton more). If the cost increases, reject
the step and **increase** $\lambda$ (be more cautious).

### 5.3 Connection to Trust Region

LM is equivalent to solving:

$$
\min_\delta \|\mathbf{r} + J\delta\|^2 \quad \text{s.t.} \quad \|\delta\| \leq \Delta
$$

where the trust region radius $\Delta$ is implicitly set by $\lambda$.

### 5.4 Convergence

- Gauss-Newton: **quadratic** convergence near the solution (when $J^TJ$ is a good
  Hessian approximation)
- LM: **superlinear** convergence; more robust to bad initial guesses
- Both require a reasonable initialisation — hence the careful SfM pipeline

In [ ]:
def levenberg_marquardt_demo(r_func, J_func, x0, max_iter=50, lam0=1e-3):
    """Minimal LM implementation for demonstration."""
    x = x0.copy()
    lam = lam0
    costs = []
    
    for iteration in range(max_iter):
        r = r_func(x)
        cost = 0.5 * np.sum(r**2)
        costs.append(cost)
        
        J = J_func(x)
        JtJ = J.T @ J
        Jtr = J.T @ r
        
        delta = np.linalg.solve(JtJ + lam * np.eye(len(x)), -Jtr)
        
        r_new = r_func(x + delta)
        cost_new = 0.5 * np.sum(r_new**2)
        
        if cost_new < cost:
            x = x + delta
            lam *= 0.5
        else:
            lam *= 2.0
        
        if abs(cost_new - cost) < 1e-12:
            break
    
    costs.append(0.5 * np.sum(r_func(x)**2))
    return x, costs


a_true, b_true = 3.0, 0.5
rng = np.random.RandomState(42)
t_data = np.linspace(0, 4, 50)
y_data = a_true * np.exp(-b_true * t_data) + rng.randn(50) * 0.1

def residuals(params):
    a, b = params
    return a * np.exp(-b * t_data) - y_data

def jacobian(params):
    a, b = params
    J = np.zeros((len(t_data), 2))
    J[:, 0] = np.exp(-b * t_data)
    J[:, 1] = -a * t_data * np.exp(-b * t_data)
    return J

x0 = np.array([1.0, 0.1])
x_opt, costs = levenberg_marquardt_demo(residuals, jacobian, x0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(costs, 'b-o', markersize=4)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cost (log scale)')
ax1.set_title('Levenberg-Marquardt Convergence')
ax1.grid(True, alpha=0.3)

ax2.scatter(t_data, y_data, s=20, c='gray', label='Noisy data')
t_fine = np.linspace(0, 4, 200)
ax2.plot(t_fine, x0[0] * np.exp(-x0[1] * t_fine), 'r--', label=f'Initial: a={x0[0]:.1f}, b={x0[1]:.1f}')
ax2.plot(t_fine, x_opt[0] * np.exp(-x_opt[1] * t_fine), 'b-', linewidth=2,
         label=f'Optimised: a={x_opt[0]:.3f}, b={x_opt[1]:.3f}')
ax2.plot(t_fine, a_true * np.exp(-b_true * t_fine), 'g:', linewidth=2,
         label=f'True: a={a_true}, b={b_true}')
ax2.set_xlabel('t')
ax2.set_ylabel('y')
ax2.set_title('Curve Fitting with Levenberg-Marquardt')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"True parameters:      a = {a_true}, b = {b_true}")
print(f"Optimised parameters: a = {x_opt[0]:.6f}, b = {x_opt[1]:.6f}")

---
## 6. Mini SfM on 5 Synthetic Views

### 6.1 Synthetic Scene Generation

We create a synthetic room with corners and objects, then place 5 cameras
around it. This provides ground truth for evaluation.

In [ ]:
def generate_synthetic_room():
    """Generate a synthetic 3D room with corners, walls, and objects."""
    points = []
    
    room_w, room_h, room_d = 6.0, 4.0, 8.0
    
    rng = np.random.RandomState(42)
    
    n_floor = 40
    floor_x = rng.uniform(-room_w/2, room_w/2, n_floor)
    floor_z = rng.uniform(2, room_d, n_floor)
    floor_y = np.full(n_floor, room_h/2)
    points.append(np.stack([floor_x, floor_y, floor_z], axis=1))
    
    n_ceil = 20
    ceil_x = rng.uniform(-room_w/2, room_w/2, n_ceil)
    ceil_z = rng.uniform(2, room_d, n_ceil)
    ceil_y = np.full(n_ceil, -room_h/2)
    points.append(np.stack([ceil_x, ceil_y, ceil_z], axis=1))
    
    for wall_x in [-room_w/2, room_w/2]:
        n_wall = 25
        wy = rng.uniform(-room_h/2, room_h/2, n_wall)
        wz = rng.uniform(2, room_d, n_wall)
        wx = np.full(n_wall, wall_x)
        points.append(np.stack([wx, wy, wz], axis=1))
    
    n_back = 30
    bx = rng.uniform(-room_w/2, room_w/2, n_back)
    by = rng.uniform(-room_h/2, room_h/2, n_back)
    bz = np.full(n_back, room_d)
    points.append(np.stack([bx, by, bz], axis=1))
    
    corners = []
    for x in [-room_w/2, room_w/2]:
        for y in [-room_h/2, room_h/2]:
            for z in [2.0, room_d]:
                corners.append([x, y, z])
    points.append(np.array(corners))
    
    table_center = np.array([0.0, 1.0, 5.0])
    n_table = 20
    table_pts = table_center + rng.randn(n_table, 3) * np.array([0.5, 0.2, 0.5])
    points.append(table_pts)
    
    box_center = np.array([-1.5, 0.5, 4.0])
    n_box = 15
    box_pts = box_center + rng.randn(n_box, 3) * np.array([0.3, 0.5, 0.3])
    points.append(box_pts)
    
    sphere_center = np.array([1.5, 0.0, 6.0])
    n_sphere = 15
    sphere_pts = sphere_center + rng.randn(n_sphere, 3) * 0.4
    points.append(sphere_pts)
    
    all_points = np.vstack(points)
    return all_points


def generate_camera_poses(n_views=5):
    """Generate n camera poses looking at the scene center."""
    scene_center = np.array([0.0, 0.0, 5.0])
    radius = 3.0
    
    poses = []
    for i in range(n_views):
        angle = -30 + i * 15  # degrees, spread from -30 to +30
        theta = np.radians(angle)
        
        cam_x = radius * np.sin(theta)
        cam_z = scene_center[2] - radius * np.cos(theta)
        cam_y = -0.5 + 0.2 * np.sin(theta * 2)
        cam_pos = np.array([cam_x, cam_y, cam_z])
        
        forward = scene_center - cam_pos
        forward = forward / np.linalg.norm(forward)
        
        up = np.array([0, -1, 0], dtype=np.float64)
        right = np.cross(forward, up)
        right = right / np.linalg.norm(right)
        up = np.cross(right, forward)
        
        R = np.stack([right, -up, forward], axis=0)
        t = -R @ cam_pos
        
        poses.append((R.astype(np.float64), t.astype(np.float64)))
    
    return poses


scene_points = generate_synthetic_room()
camera_poses = generate_camera_poses(n_views=5)

print(f"Scene: {len(scene_points)} 3D points")
print(f"Cameras: {len(camera_poses)} views")
print(f"Scene bounding box: [{scene_points.min(axis=0)}] to [{scene_points.max(axis=0)}]")

In [ ]:
def visualize_scene_and_cameras(points, poses, K, title='Scene and Cameras'):
    """Visualize the 3D scene and camera positions."""
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    ax.scatter(points[:, 0], points[:, 2], -points[:, 1],
               c='gray', s=8, alpha=0.5, label='Scene points')
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(poses)))
    cam_scale = 0.3
    
    for i, (R, t) in enumerate(poses):
        cam_pos = -R.T @ t
        
        axes_world = R.T @ (np.eye(3) * cam_scale)
        
        ax.scatter(cam_pos[0], cam_pos[2], -cam_pos[1],
                   s=100, c=[colors[i]], marker='s', zorder=5)
        ax.text(cam_pos[0], cam_pos[2], -cam_pos[1] + 0.3,
                f'Cam {i}', fontsize=9, color=colors[i])
        
        for axis_idx, c in enumerate([(1,0,0), (0,1,0), (0,0,1)]):
            end = cam_pos + axes_world[:, axis_idx]
            ax.plot([cam_pos[0], end[0]], [cam_pos[2], end[2]],
                    [-cam_pos[1], -end[1]], color=c, linewidth=2)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Z')
    ax.set_zlabel('-Y')
    ax.set_title(title, fontsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

visualize_scene_and_cameras(scene_points, camera_poses, K_sfm,
                             title='Synthetic Room Scene + 5 Camera Views')

In [ ]:
def generate_observations(scene_points, camera_poses, K, noise_std=1.0, seed=42):
    """
    Project scene points into each camera and generate noisy observations.
    Returns observations and visibility information.
    """
    rng = np.random.RandomState(seed)
    n_views = len(camera_poses)
    n_points = len(scene_points)
    W, H = 640, 480
    
    observations = {}  # (view_idx, point_idx) → (u, v)
    view_keypoints = {}  # view_idx → list of (kp_2d, point_idx)
    
    for vi, (R, t) in enumerate(camera_poses):
        view_keypoints[vi] = []
        for pi in range(n_points):
            X_cam = R @ scene_points[pi] + t
            
            if X_cam[2] < 0.1:
                continue
            
            u = K[0, 0] * X_cam[0] / X_cam[2] + K[0, 2]
            v = K[1, 1] * X_cam[1] / X_cam[2] + K[1, 2]
            
            margin = 10
            if margin <= u < W - margin and margin <= v < H - margin:
                u_noisy = u + rng.randn() * noise_std
                v_noisy = v + rng.randn() * noise_std
                observations[(vi, pi)] = np.array([u_noisy, v_noisy])
                view_keypoints[vi].append((np.array([u_noisy, v_noisy]), pi))
    
    print(f"Total observations: {len(observations)}")
    for vi in range(n_views):
        n_vis = len(view_keypoints[vi])
        print(f"  View {vi}: {n_vis} visible points")
    
    return observations, view_keypoints


observations, view_keypoints = generate_observations(
    scene_points, camera_poses, K_sfm, noise_std=1.0
)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for vi in range(5):
    ax = axes[vi]
    kps = view_keypoints[vi]
    if kps:
        pts = np.array([kp[0] for kp in kps])
        ax.scatter(pts[:, 0], pts[:, 1], s=5, c='blue', alpha=0.7)
    ax.set_xlim(0, 640)
    ax.set_ylim(480, 0)
    ax.set_title(f'View {vi} ({len(kps)} pts)', fontsize=10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
plt.suptitle('Projected Keypoints in Each Camera View', fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Triangulation — DLT from the Cross-Product Constraint

Given two camera matrices $P_1, P_2$ and corresponding 2-D points $\mathbf{x}_1, \mathbf{x}_2$,
**DLT triangulation** recovers the 3-D point $\mathbf{X}$.

#### Step 1: The Cross-Product Constraint

If $\mathbf{x} = P \mathbf{X}$ (in homogeneous coordinates, i.e. up to scale), then
$\mathbf{x}$ and $P\mathbf{X}$ are parallel, which means their cross product vanishes:

$$
\boxed{\mathbf{x} \times (P \mathbf{X}) = \mathbf{0}}
$$

#### Step 2: Expanding the Cross Product

Write $\mathbf{x} = [u, v, 1]^\top$ and denote the rows of $P$ as
$\mathbf{p}^{1\top}, \mathbf{p}^{2\top}, \mathbf{p}^{3\top} \in \mathbb{R}^{1 \times 4}$,
so $P\mathbf{X} = [\mathbf{p}^{1\top}\mathbf{X},\; \mathbf{p}^{2\top}\mathbf{X},\; \mathbf{p}^{3\top}\mathbf{X}]^\top$.

The cross product $\mathbf{x} \times (P\mathbf{X})$ gives three equations:

$$
\mathbf{x} \times (P\mathbf{X}) = \begin{bmatrix}
v \cdot \mathbf{p}^{3\top}\mathbf{X} - 1 \cdot \mathbf{p}^{2\top}\mathbf{X} \\
1 \cdot \mathbf{p}^{1\top}\mathbf{X} - u \cdot \mathbf{p}^{3\top}\mathbf{X} \\
u \cdot \mathbf{p}^{2\top}\mathbf{X} - v \cdot \mathbf{p}^{1\top}\mathbf{X}
\end{bmatrix} = \mathbf{0}
$$

Each row is **linear in $\mathbf{X}$**. However, only **2 of 3 are linearly independent**:
the third equation equals $u \times (\text{eq. 1}) + v \times (\text{eq. 2})$.
We take the first two from each view:

$$
\text{View } k: \quad
\begin{cases}
u_k \, \mathbf{p}_k^{3\top}\mathbf{X} - \mathbf{p}_k^{1\top}\mathbf{X} = 0 \\
v_k \, \mathbf{p}_k^{3\top}\mathbf{X} - \mathbf{p}_k^{2\top}\mathbf{X} = 0
\end{cases}
$$

#### Step 3: Assembling the Linear System

Stacking 2 equations from each of 2 views gives the $4 \times 4$ homogeneous system $A\mathbf{X} = \mathbf{0}$:

$$
\boxed{
A = \begin{bmatrix}
u_1 \mathbf{p}_1^{3\top} - \mathbf{p}_1^{1\top} \\
v_1 \mathbf{p}_1^{3\top} - \mathbf{p}_1^{2\top} \\
u_2 \mathbf{p}_2^{3\top} - \mathbf{p}_2^{1\top} \\
v_2 \mathbf{p}_2^{3\top} - \mathbf{p}_2^{2\top}
\end{bmatrix} \in \mathbb{R}^{4 \times 4}
}
$$

With $N > 2$ views, each view contributes 2 rows, giving a $2N \times 4$ overdetermined system.

#### Step 4: SVD Solution

Since $A\mathbf{X} = \mathbf{0}$ is a homogeneous system (the scale of $\mathbf{X}$ is arbitrary
in homogeneous coordinates), we seek the non-trivial $\mathbf{X}$ that minimises
$\|A\mathbf{X}\|^2$ subject to $\|\mathbf{X}\| = 1$.

Compute the SVD: $A = U \Sigma V^\top$. The solution is the **right singular vector
corresponding to the smallest singular value** — i.e. the last column of $V$
(equivalently, the last row of $V^\top$).

**Why SVD works:** By the Eckart–Young theorem, $\min_{\|\mathbf{X}\|=1} \|A\mathbf{X}\|^2 = \sigma_{\min}^2$,
and the minimiser is the corresponding right singular vector. In the noise-free case,
$\sigma_{\min} = 0$ and $\mathbf{X}$ lies exactly in the null space of $A$.

Finally, **dehomogenise**: $\mathbf{X}_{3D} = [X/W,\; Y/W,\; Z/W]^\top$ where
$\mathbf{X} = [X, Y, Z, W]^\top$.

In [ ]:
def triangulate_dlt(pt1, pt2, P1, P2):
    """DLT triangulation for a pair of 2D correspondences."""
    A = np.zeros((4, 4))
    A[0] = pt1[0] * P1[2] - P1[0]
    A[1] = pt1[1] * P1[2] - P1[1]
    A[2] = pt2[0] * P2[2] - P2[0]
    A[3] = pt2[1] * P2[2] - P2[1]
    _, _, Vt = np.linalg.svd(A)
    X = Vt[-1]
    return X[:3] / X[3]


def triangulate_points(observations, view_keypoints, camera_poses, K, n_points):
    """Triangulate all points visible in at least 2 views."""
    n_views = len(camera_poses)
    
    point_views = {}  # point_idx → list of (view_idx, 2d_pt)
    for (vi, pi), pt2d in observations.items():
        if pi not in point_views:
            point_views[pi] = []
        point_views[pi].append((vi, pt2d))
    
    triangulated = {}
    for pi, view_list in point_views.items():
        if len(view_list) < 2:
            continue
        
        vi1, pt1 = view_list[0]
        vi2, pt2 = view_list[1]
        R1, t1 = camera_poses[vi1]
        R2, t2 = camera_poses[vi2]
        P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
        P2 = K @ np.hstack([R2, t2.reshape(3, 1)])
        
        X = triangulate_dlt(pt1, pt2, P1, P2)
        triangulated[pi] = X
    
    return triangulated


triangulated_points = triangulate_points(
    observations, view_keypoints, camera_poses, K_sfm, len(scene_points)
)

print(f"Triangulated {len(triangulated_points)} / {len(scene_points)} points")

tri_pts = np.array(list(triangulated_points.values()))
tri_indices = list(triangulated_points.keys())
gt_pts = scene_points[tri_indices]

errors = np.linalg.norm(tri_pts - gt_pts, axis=1)
print(f"\nTriangulation error:")
print(f"  Mean:   {errors.mean():.4f} m")
print(f"  Median: {np.median(errors):.4f} m")
print(f"  Max:    {errors.max():.4f} m")

### 6.3 Bundle Adjustment Implementation

We use `scipy.optimize.least_squares` with Levenberg–Marquardt to jointly refine all camera
parameters (axis-angle rotation + translation) and all 3-D point positions. After optimisation,
estimated points are aligned to ground truth via Procrustes analysis for fair error evaluation.

In [ ]:
def run_bundle_adjustment(camera_poses_init, points_3d_init, observations_ba, K,
                          max_iterations=100):
    """
    Run bundle adjustment using scipy's least_squares (Levenberg-Marquardt).
    
    Parameterisation:
    - Each camera: 6 params (axis-angle rotation + translation)
    - Each point: 3 params (X, Y, Z)
    
    The 7-DOF gauge ambiguity (rotation + translation + scale) is implicitly
    resolved by LM damping, which keeps the solution near the initialisation.
    After optimisation the result is aligned to ground truth via Procrustes
    for fair error evaluation.
    
    Cost: Σ ||π(K[R|t]X) - x_obs||²
    """
    n_cameras = len(camera_poses_init)
    n_points = len(points_3d_init)

    cam_params = np.zeros(6 * n_cameras)
    for i, (R, t) in enumerate(camera_poses_init):
        rvec, _ = cv2.Rodrigues(R)
        cam_params[6*i:6*i+3] = rvec.ravel()
        cam_params[6*i+3:6*i+6] = t

    point_params = np.array(points_3d_init).ravel()
    x0 = np.concatenate([cam_params, point_params])

    obs_list = []
    for (vi, pi), pt2d in observations_ba.items():
        obs_list.append((vi, pi, pt2d))

    def residual_func(x):
        cams = x[:6 * n_cameras].reshape(n_cameras, 6)
        pts = x[6 * n_cameras:].reshape(n_points, 3)

        residuals_vec = np.zeros(2 * len(obs_list))
        for k, (vi, pi, pt2d) in enumerate(obs_list):
            R_k, _ = cv2.Rodrigues(cams[vi, :3])
            X_cam = R_k @ pts[pi] + cams[vi, 3:6]
            Z_c = max(X_cam[2], 0.01)
            proj_2d = np.array([
                K[0, 0] * X_cam[0] / Z_c + K[0, 2],
                K[1, 1] * X_cam[1] / Z_c + K[1, 2],
            ])
            residuals_vec[2*k:2*k+2] = proj_2d - pt2d
        return residuals_vec

    r0 = residual_func(x0)
    reproj_before = np.sqrt(np.mean(r0**2))

    result = least_squares(residual_func, x0, method='lm',
                           max_nfev=max_iterations * len(x0))

    r_final = residual_func(result.x)
    reproj_after = np.sqrt(np.mean(r_final**2))

    cams_opt = result.x[:6 * n_cameras].reshape(n_cameras, 6)
    pts_opt = result.x[6 * n_cameras:].reshape(n_points, 3)

    camera_poses_opt = []
    for i in range(n_cameras):
        R_opt, _ = cv2.Rodrigues(cams_opt[i, :3])
        t_opt = cams_opt[i, 3:6]
        camera_poses_opt.append((R_opt, t_opt))

    return camera_poses_opt, pts_opt, reproj_before, reproj_after


def procrustes_align(pts_est, pts_gt):
    """Align estimated points to GT via similarity transform (Procrustes)."""
    mu_e = pts_est.mean(axis=0)
    mu_g = pts_gt.mean(axis=0)
    de = pts_est - mu_e
    dg = pts_gt - mu_g
    s = np.sqrt(np.sum(dg**2) / max(np.sum(de**2), 1e-12))
    U, _, Vt = np.linalg.svd(de.T @ dg)
    D = np.eye(3)
    D[2, 2] = np.linalg.det(Vt.T @ U.T)
    R = Vt.T @ D @ U.T
    return s * (R @ de.T).T + mu_g, s, R, mu_e, mu_g


rng_ba = np.random.RandomState(123)
noisy_poses = []
for R, t in camera_poses:
    R_noise = Rotation.from_rotvec(rng_ba.randn(3) * 0.01).as_matrix()
    t_noise = t + rng_ba.randn(3) * 0.02
    noisy_poses.append((R_noise @ R, t_noise))

tri_pts_noisy = tri_pts + rng_ba.randn(*tri_pts.shape) * 0.05

obs_ba = {}
point_idx_map = {old_idx: new_idx for new_idx, old_idx in enumerate(tri_indices)}
for (vi, pi), pt2d in observations.items():
    if pi in point_idx_map:
        obs_ba[(vi, point_idx_map[pi])] = pt2d

print("Running bundle adjustment...")
poses_opt, pts_opt, reproj_before, reproj_after = run_bundle_adjustment(
    noisy_poses, tri_pts_noisy, obs_ba, K_sfm
)

print(f"\nReprojection error:")
print(f"  Before BA: {reproj_before:.4f} px")
print(f"  After BA:  {reproj_after:.4f} px")

gt_pts_ordered = scene_points[tri_indices]

pts_before_aligned, _, _, _, _ = procrustes_align(tri_pts_noisy, gt_pts_ordered)
pts_after_aligned, s_align, R_align, mu_e, mu_g = procrustes_align(pts_opt, gt_pts_ordered)

errors_before = np.linalg.norm(pts_before_aligned - gt_pts_ordered, axis=1)
errors_after = np.linalg.norm(pts_after_aligned - gt_pts_ordered, axis=1)

poses_opt_aligned = []
for R_o, t_o in poses_opt:
    cam_pos = -R_o.T @ t_o
    cam_pos_a = s_align * (R_align @ (cam_pos - mu_e)) + mu_g
    R_a = R_o @ R_align.T
    t_a = -R_a @ cam_pos_a
    poses_opt_aligned.append((R_a, t_a))
poses_opt = poses_opt_aligned

print(f"\n3D point error (Procrustes-aligned):")
print(f"  Before BA: mean = {errors_before.mean():.4f} m, median = {np.median(errors_before):.4f} m")
print(f"  After BA:  mean = {errors_after.mean():.4f} m, median = {np.median(errors_after):.4f} m")

In [ ]:
fig = plt.figure(figsize=(18, 6))

titles = ['Ground Truth', 'Before BA (noisy)', 'After BA (optimised)']
point_sets = [gt_pts_ordered, tri_pts_noisy, pts_opt]
pose_sets = [camera_poses, noisy_poses, poses_opt]

for idx in range(3):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    pts = point_sets[idx]
    ax.scatter(pts[:, 0], pts[:, 2], -pts[:, 1], c='gray', s=5, alpha=0.5)
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(pose_sets[idx])))
    for i, (R, t) in enumerate(pose_sets[idx]):
        cam_pos = -R.T @ t
        ax.scatter(cam_pos[0], cam_pos[2], -cam_pos[1],
                   s=80, c=[colors[i]], marker='s', zorder=5)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Z')
    ax.set_zlabel('-Y')
    ax.set_title(titles[idx], fontsize=12)

plt.suptitle('Bundle Adjustment: Before and After', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(errors_before, bins=30, alpha=0.6, color='red', label='Before BA')
ax1.hist(errors_after, bins=30, alpha=0.6, color='green', label='After BA')
ax1.set_xlabel('3D Point Error (m)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of 3D Point Errors')
ax1.legend()
ax1.grid(True, alpha=0.3)

rot_errors_before = []
rot_errors_after = []
trans_errors_before = []
trans_errors_after = []

for i, (R_gt, t_gt) in enumerate(camera_poses):
    R_noisy, t_noisy = noisy_poses[i]
    R_opt, t_opt = poses_opt[i]
    
    rot_err_b = np.degrees(np.arccos(np.clip((np.trace(R_noisy @ R_gt.T) - 1)/2, -1, 1)))
    rot_err_a = np.degrees(np.arccos(np.clip((np.trace(R_opt @ R_gt.T) - 1)/2, -1, 1)))
    rot_errors_before.append(rot_err_b)
    rot_errors_after.append(rot_err_a)
    
    trans_errors_before.append(np.linalg.norm(t_noisy - t_gt))
    trans_errors_after.append(np.linalg.norm(t_opt - t_gt))

x_pos = np.arange(len(camera_poses))
width = 0.35
ax2.bar(x_pos - width/2, rot_errors_before, width, label='Before BA', color='red', alpha=0.7)
ax2.bar(x_pos + width/2, rot_errors_after, width, label='After BA', color='green', alpha=0.7)
ax2.set_xlabel('Camera Index')
ax2.set_ylabel('Rotation Error (degrees)')
ax2.set_title('Camera Rotation Errors')
ax2.set_xticks(x_pos)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Camera pose errors:")
print(f"{'':>8} {'Rot Before':>12} {'Rot After':>12} {'Trans Before':>14} {'Trans After':>14}")
for i in range(len(camera_poses)):
    print(f"Cam {i}:  {rot_errors_before[i]:>10.4f}°  {rot_errors_after[i]:>10.4f}°"
          f"  {trans_errors_before[i]:>12.6f} m  {trans_errors_after[i]:>12.6f} m")

---
## 7. COLMAP and Multi-View Stereo (MVS)

### 7.1 COLMAP

COLMAP (Schönberger & Frahm, CVPR 2016) is the gold-standard SfM system.
Key innovations beyond basic incremental SfM:

1. **Robust initialisation**: selects the best pair using the number of
   inliers, the distribution of feature matches, and the triangulation angle

2. **Next-best-view selection**: registers images in order of most
   2D-3D correspondences (greedy approach for robustness)

3. **Iterative refinement**: runs BA after every new image registration,
   plus global BA periodically

4. **Outlier filtering**: removes points with high reprojection error
   or poor triangulation angles

### 7.2 COLMAP Pipeline

```
colmap feature_extractor --database_path db.db --image_path images/
colmap exhaustive_matcher --database_path db.db
colmap mapper --database_path db.db --image_path images/ --output_path sparse/
```

### 7.3 Multi-View Stereo (MVS)

SfM produces a **sparse** point cloud. To obtain a **dense** reconstruction:

1. **Depth map estimation**: for each image, estimate a dense depth map using
   multi-view photometric consistency:

$$
C(\mathbf{p}, d) = 1 - \text{NCC}\bigl(I_{\text{ref}}(W_p), I_{\text{src}}(W_{\pi(\mathbf{p}, d)})\bigr)
$$

   where $W_p$ denotes the $k \times k$ image patch centred at pixel $\mathbf{p}$, and NCC is normalised cross-correlation.

2. **Depth map fusion**: merge per-view depth maps into a consistent global
   point cloud, filtering outliers via cross-view consistency checks.

3. **Surface reconstruction**: Poisson reconstruction or Delaunay-based
   meshing from the dense point cloud.

### 7.4 Neural 3D Reconstruction

Modern approaches replace or augment MVS with neural methods:

| Method | Input | Output | Key Idea |
|:---|:---|:---|:---|
| NeRF | Posed images | Neural radiance field | Volume rendering, MLP |
| 3D Gaussian Splatting | Posed images + SfM points | Gaussian primitives | Differentiable rasterization |
| Instant NGP | Posed images | Hash-grid NeRF | Multi-resolution hash encoding |
| NeuS | Posed images | Neural SDF | Volume rendering of signed distance |

---
## 7b. Global SfM

Incremental SfM (Section 2, COLMAP) adds cameras one at a time. **Global SfM**
solves for all camera poses simultaneously, avoiding the ordering-dependent
error accumulation of the incremental approach.

### 7b.1 Rotation Averaging

Given pairwise relative rotations $R_{ij}$ from the essential matrix
decomposition of each image pair, find **absolute** rotations $\{R_i\}$ such
that $R_{ij} \approx R_j R_i^{-1}$.

This is an optimisation problem on $\text{SO}(3)^n$:

$$
\min_{\{R_i\}} \sum_{(i,j) \in \mathcal{E}} d_{\text{geo}}(R_{ij},\; R_j R_i^T)
$$

where $d_{\text{geo}}(R_1, R_2) = \|\log(R_1^T R_2)\|$ is the geodesic distance
on SO(3).

> where $\|\cdot\|$ is the Riemannian norm on $\mathfrak{so}(3)$, equivalent to $\arccos\bigl(\frac{\operatorname{trace}(R_1^\top R_2) - 1}{2}\bigr)$ — the rotation angle between $R_1$ and $R_2$.

> **Jacobian of the geodesic residual.** Define $M = R_{ij}^\top R_j R_i^\top$ and $\mathbf{r}_{ij} = \text{Log}(M) \in \mathbb{R}^3$. Under right perturbation $R_k \leftarrow R_k \cdot \text{Exp}(\delta\boldsymbol{\phi}_k)$:
>
> Here $J_r(\mathbf{r})$ is the **right Jacobian** of SO(3), defined by $\text{Exp}(\mathbf{r} + \delta\mathbf{r}) \approx \text{Exp}(\mathbf{r})\,\text{Exp}(J_r(\mathbf{r})^{-1}\delta\mathbf{r})$, with closed form $J_r(\mathbf{r}) = I - \frac{1-\cos\theta}{\theta^2}[\mathbf{r}]_\times + \frac{\theta - \sin\theta}{\theta^3}[\mathbf{r}]_\times^2$ where $\theta = \|\mathbf{r}\|$. Its inverse $J_r^{-1}(\mathbf{r})$ appears below via the BCH approximation $\text{Log}(M\,\text{Exp}(\delta\psi)) \approx \mathbf{r} + J_r^{-1}(\mathbf{r})\,\delta\psi$.
>
> **For $R_j$:** $M' = R_{ij}^\top R_j \text{Exp}(\delta\phi_j) R_i^\top$. Factor out $M$:
>
> $$M' = M \cdot \underbrace{(R_i R_j^\top R_{ij})}_{M^{-1}} R_{ij}^\top R_j \text{Exp}(\delta\phi_j) R_i^\top = M \cdot R_i \text{Exp}(\delta\phi_j) R_i^\top = M \cdot \text{Exp}(R_i\,\delta\phi_j)$$
>
> using the SO(3) adjoint $R\,\text{Exp}(\phi)\,R^\top = \text{Exp}(R\phi)$. Applying the BCH approximation $\text{Log}(M \text{Exp}(\delta\psi)) \approx \mathbf{r} + J_r^{-1}(\mathbf{r})\,\delta\psi$:
>
> $$\boxed{\frac{\partial \mathbf{r}_{ij}}{\partial \delta\boldsymbol{\phi}_j} = J_r^{-1}(\mathbf{r}_{ij})\,R_i}$$
>
> **For $R_i$:** $(R_i \text{Exp}(\delta\phi_i))^\top = \text{Exp}(-\delta\phi_i) R_i^\top$, so $M' = R_{ij}^\top R_j \text{Exp}(-\delta\phi_i) R_i^\top$. By the same factoring:
>
> $$M' = M \cdot R_i \text{Exp}(-\delta\phi_i) R_i^\top = M \cdot \text{Exp}(-R_i\,\delta\phi_i)$$
>
> $$\boxed{\frac{\partial \mathbf{r}_{ij}}{\partial \delta\boldsymbol{\phi}_i} = -J_r^{-1}(\mathbf{r}_{ij})\,R_i}$$
>
> Near convergence ($\mathbf{r}_{ij} \to 0$), $J_r^{-1} \to I$ and the Jacobians simplify to $\pm R_i$.

**L1 rotation averaging** (Chatterjee & Govindu, ICCV 2013) minimises the sum
of geodesic distances rather than squared distances, making it **robust to
outlier relative rotations** from incorrect epipolar estimates. The L1 formulation
is solved via iteratively reweighted least squares (IRLS).

### 7b.2 Translation Averaging

Given pairwise translation **directions** $\hat{\mathbf{t}}_{ij}$ (unit vectors —
the scale of each pairwise translation is unknown) and the solved absolute
rotations, find absolute camera positions $\{\mathbf{c}_i\}$ such that:

$$
\hat{\mathbf{t}}_{ij} \approx \frac{\mathbf{c}_j - \mathbf{c}_i}{\|\mathbf{c}_j - \mathbf{c}_i\|}
$$

This is a **1DOF problem per edge** (known direction, unknown scale) and can be
formulated as a linear system per coordinate or solved via second-order cone
programming (SOCP).

### 7b.3 Advantages and Disadvantages

| Property | Global SfM | Incremental SfM (COLMAP) |
|:---|:---|:---|
| Error accumulation | None (all cameras solved jointly) | Drift from ordering |
| Speed | Fast ($O(n)$ after pairwise matching) | Slow (repeated BA) |
| Outlier robustness | Sensitive (relies on pairwise estimates) | Robust (RANSAC + iterative BA) |
| Degeneracies | Harder to detect/handle | Gracefully skips bad images |

Global SfM is preferred for **large-scale, well-connected** image collections
(e.g., internet photo tourism). Incremental SfM remains more robust for
challenging, sparsely connected datasets.

---
## 7c. PatchMatch Multi-View Stereo

Section 7.3 described MVS generically. The specific algorithm behind COLMAP's
`patch_match_stereo` command is **PatchMatch Stereo** (Galliani et al., 2015;
Zheng et al., 2014), a randomised, iterative depth estimation algorithm.

### 7c.1 Per-Pixel Hypothesis

For each pixel $\mathbf{p}$ in the reference image, maintain a hypothesis
$(d, \mathbf{n})$: depth $d \in [d_{\min}, d_{\max}]$ and surface normal
$\mathbf{n} \in \mathbb{S}^2$.

### 7c.2 The Three-Step Iteration

Each PatchMatch iteration cycles through three stages:

**(a) Random initialisation** (iteration 0 only): assign each pixel a random
depth $d \sim \mathcal{U}(d_{\min}, d_{\max})$ and a random unit normal.

**(b) Spatial propagation**: for each pixel, check whether a **neighbour's**
hypothesis gives a lower photometric cost at your pixel. If so, adopt it.
Good hypotheses "spread" spatially across the image, converging in 3–5
iterations. Propagation alternates direction (top-left → bottom-right, then
bottom-right → top-left) each iteration.

**(c) Random refinement**: perturb the current hypothesis by sampling
from progressively smaller ranges:

$$
d' \sim \mathcal{U}(d - \Delta_k, d + \Delta_k), \qquad \Delta_k = \Delta_0 / 2^k
$$

where $k$ is the refinement step. Similarly for the normal. This explores
the neighbourhood of the current best hypothesis with decreasing step size.

### 7c.3 Photometric Cost

Given a depth + normal hypothesis at pixel $\mathbf{p}$, we warp a local patch
from the reference image into each source view using the induced homography:

$$
H = K_s \left(R_{sr} - \frac{\mathbf{t}_{sr} \mathbf{n}^T}{d}\right) K_r^{-1}
$$

The photometric cost is the **normalised cross-correlation (NCC)** between
the reference patch and the warped source patch:

$$
C(\mathbf{p}, d, \mathbf{n}) = 1 - \text{NCC}\bigl(W_{\text{ref}}(\mathbf{p}),\; W_{\text{src}}(H \cdot \mathbf{p})\bigr)
$$

### 7c.4 Automatic View Selection

Not all source views are equally useful. COLMAP selects the best source views
per reference image based on:
- **Triangulation angle**: too small → poor depth resolution; too large → patch
  distortion. Optimal: 5°–20°.
- **Overlap**: sufficient shared scene visibility.

### 7c.5 Geometric Consistency Filtering

After all depth maps converge, a cross-view consistency check removes outliers:

1. Project pixel $\mathbf{p}$ with depth $d$ from reference view into source view
2. Read the source view's depth at the projected location
3. Re-project back to the reference view
4. If the round-trip reprojection error $< \epsilon$ and depth discrepancy $< \tau$,
   the depth is **geometrically consistent**

Only depths consistent across $\geq k$ views (typically $k = 2$) are kept.

### 7c.6 COLMAP MVS Commands

```
colmap patch_match_stereo \
    --workspace_path dense/ --workspace_format COLMAP
colmap stereo_fusion \
    --workspace_path dense/ --output_path dense/fused.ply
```

---
## 7d. OpenMVS — Alternative Dense Reconstruction

**OpenMVS** (Open Multi-View Stereo) is an open-source library that takes
COLMAP's **sparse** reconstruction as input and produces a dense textured mesh.
It is a popular alternative to COLMAP's built-in MVS, often faster on GPU
hardware with comparable quality.

### Pipeline

```
DensifyPointCloud  →  ReconstructMesh  →  RefineMesh  →  TextureMesh
```

| Stage | What it does |
|:---|:---|
| `DensifyPointCloud` | PatchMatch-style depth estimation + fusion → dense point cloud |
| `ReconstructMesh` | Delaunay tetrahedralisation + graph-cut labelling → watertight mesh |
| `RefineMesh` | Photometric + smoothness optimisation of vertex positions |
| `TextureMesh` | Per-face view selection + Poisson blending → textured mesh |

### Key Features

- **CUDA acceleration** for depth map computation — significantly faster than
  CPU-only COLMAP MVS on large datasets
- Reads COLMAP's sparse output directly (`model_convert` to OpenMVS format)
- Produces publication-quality textured meshes with automatic view selection

### When to Use OpenMVS vs COLMAP MVS

| Criterion | COLMAP MVS | OpenMVS |
|:---|:---|:---|
| GPU support | Limited | Full CUDA |
| Speed (large scenes) | Slower | Faster |
| Mesh quality | Point cloud only (needs Poisson) | Full mesh pipeline |
| Integration | Built-in | Separate tool |

A typical production workflow: **COLMAP** for sparse SfM →
**OpenMVS** for dense reconstruction and meshing.

---
## 7e. The End of Classical SfM? Foundation Models for 3D (2024–2026)

The classical SfM pipeline — detect, match, triangulate, bundle-adjust — has been the standard for
two decades. Starting in 2024, a new family of methods replaces the entire pipeline with a single
feed-forward neural network.

### 7e.1 DUSt3R and MASt3R

**DUSt3R** (Wang et al., CVPR 2024) reframes two-view reconstruction as dense regression: given
two images, a ViT-based encoder-decoder directly predicts per-pixel 3D pointmaps
$\hat{X}^{(1)}, \hat{X}^{(2)} \in \mathbb{R}^{H \times W \times 3}$ aligned in the first camera's frame. No keypoints,
no RANSAC, no explicit epipolar geometry. The confidence-weighted regression loss operates in 3D:

$$\mathcal{L} = \sum_{i} w_i \| \hat{X}_i - X_i^{\text{gt}} \|_1$$

**MASt3R** (Leroy et al., ECCV 2024) adds a matching head that produces dense correspondences
alongside pointmaps, enabling explicit feature matching between views.

### 7e.2 Scaling to Many Views

The pairwise nature of DUSt3R/MASt3R requires post-hoc global alignment:

| Method | Strategy | Scale | Optimization? |
|:---|:---|:---|:---|
| **MASt3R-SfM** (3DV 2025) | ASMK retrieval → sparse global alignment → reprojection refinement | 1000+ images | Yes (gradient descent) |
| **MUSt3R** (CVPR 2025) | Symmetric encoder + working memory → shared coordinate frame | 100+ images | No (feed-forward) |
| **Regist3R** (arXiv 2025) | Incremental registration via MST traversal with explicit pointmaps | 1000+ images | No (inference-only) |
| **VGGT** (CVPR 2025) | Feed-forward transformer → cameras, depth, pointmaps, tracks | 100+ images | No (single forward pass) |

### 7e.3 VGGT: Visual Geometry Grounded Transformer

VGGT (Wang et al., CVPR 2025) is particularly notable: a single transformer processes $N$ images
and outputs all of:
- Camera intrinsics and extrinsics for each view
- Dense depth maps
- Dense 3D pointmaps
- Dense inter-frame point tracks

This eliminates **every** component of the classical pipeline in a single forward pass.
On DTU and Tanks & Temples, VGGT outperforms DUSt3R and MASt3R by large margins.

### 7e.4 When Classical SfM Still Wins

Despite the impressive results, foundation models have limitations:

1. **Metric scale**: learned models produce relative reconstructions; classical stereo with known
   baseline gives true metric depth
2. **Precision**: for surveying/mapping applications requiring sub-cm accuracy, bundle adjustment
   with calibrated cameras remains superior
3. **Transparency**: classical pipelines produce interpretable intermediate results (matches,
   inlier counts, reprojection errors) that enable failure diagnosis
4. **Edge deployment**: classical pipelines run without GPU; learned models require significant
   compute

**For autonomous drones**: the hybrid approach is emerging — use MASt3R for robust initialisation
and loop closure, but maintain a classical BA backend for metric consistency and real-time
incremental updates.

---
## 8. Exercises

### Exercise 10.1: Implement DLT Triangulation

Given two camera matrices and corresponding 2-D points, triangulate the 3-D point.

In [ ]:
def exercise_triangulate(pt1, pt2, P1, P2):
    """
    Exercise 10.1: DLT triangulation.
    
    Build the 4×4 system A from:
        Row 0: u1 * P1[2] - P1[0]
        Row 1: v1 * P1[2] - P1[1]
        Row 2: u2 * P2[2] - P2[0]
        Row 3: v2 * P2[2] - P2[1]
    
    Solve via SVD: X is last row of Vt.
    Dehomogenise: X_3d = X[:3] / X[3]
    
    Returns: np.ndarray shape (3,)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement DLT triangulation")


# --- Tests ---
# R_t1 = np.eye(3)
# t_t1 = np.zeros(3)
# R_t2 = np.eye(3)
# t_t2 = np.array([-0.5, 0, 0])  # 0.5m baseline
# K_test = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1.0]])
# P1_test = K_test @ np.hstack([R_t1, t_t1.reshape(3,1)])
# P2_test = K_test @ np.hstack([R_t2, t_t2.reshape(3,1)])
# X_true = np.array([1.0, 0.5, 5.0])
# x1 = (K_test @ (R_t1 @ X_true + t_t1))
# x1 = x1[:2] / x1[2]
# x2 = (K_test @ (R_t2 @ X_true + t_t2))
# x2 = x2[:2] / x2[2]
# X_est = exercise_triangulate(x1, x2, P1_test, P2_test)
# assert np.allclose(X_est, X_true, atol=1e-6), f"Expected {X_true}, got {X_est}"
# print(f"✓ exercise_triangulate passed: {X_est}")

### Exercise 10.2: PnP Camera Registration

Given known 3-D points and their 2-D projections, estimate the camera pose.

In [ ]:
def exercise_pnp(pts_3d, pts_2d, K):
    """
    Exercise 10.2: Solve PnP for camera pose.
    
    Use cv2.solvePnPRansac to find R, t.
    
    Returns: R (3×3), t (3,), n_inliers
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement PnP")


# --- Tests ---
# Use the synthetic data from earlier:
# R_est, t_est, n_inl = exercise_pnp(pts_3d_test[valid], pts_2d_noisy[valid], K_sfm)
# rot_err = np.degrees(np.arccos(np.clip((np.trace(R_est @ R_true.T) - 1)/2, -1, 1)))
# assert rot_err < 1.0, f"Rotation error too large: {rot_err}°"
# print(f"✓ exercise_pnp passed (rot err = {rot_err:.4f}°, {n_inl} inliers)")

### Exercise 10.3: Mini SfM Pipeline

Implement a mini SfM pipeline that:
1. Initialises from views 0 and 1
2. Registers views 2, 3, 4 via PnP
3. Triangulates all points
4. Runs bundle adjustment
5. Reports final reprojection error

In [ ]:
def exercise_mini_sfm(observations, view_keypoints, K, n_views=5):
    """
    Exercise 10.3: Mini SfM pipeline.
    
    Steps:
    1. Initialise from view pair (0, 1)
       - Use essential matrix decomposition
       - Triangulate initial points
    2. For views 2, 3, 4:
       - Find 2D-3D correspondences
       - Register via PnP
       - Triangulate new points
    3. Run bundle adjustment
    
    Returns:
    - camera_poses: list of (R, t)
    - points_3d: np.ndarray (M, 3)
    - reproj_error: float
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement mini SfM")

---
## Summary

### Key Equations

| Concept | Formula |
|:---|:---|
| Reprojection error | $\mathbf{r}_{ij} = \pi(K[R_i \mid \mathbf{t}_i]\mathbf{X}_j) - \mathbf{x}_{ij}$ |
| BA objective | $\min \sum_{i,j} \|\mathbf{r}_{ij}\|^2$ |
| Gauss-Newton | $J^T J \, \delta = -J^T \mathbf{r}$ |
| Levenberg-Marquardt | $(J^T J + \lambda I) \, \delta = -J^T \mathbf{r}$ |
| Schur complement | $S = H_{cc} - H_{cp} \, H_{pp}^{-1} \, H_{pc}$ |
| DLT triangulation | $A\mathbf{X} = 0$, solved by SVD |

### Key Takeaways

1. Incremental SfM: initialise pair → register via PnP → triangulate → BA
2. PnP solves for camera pose from known 3D-2D correspondences (P3P minimal solver)
3. Bundle adjustment jointly optimises all cameras and points
4. The **Schur complement** exploits block-diagonal $H_{pp}$ to reduce the system from $O((6n+3M)^3)$ to $O((6n)^3)$
5. LM interpolates between Gauss-Newton (fast near optimum) and gradient descent (robust far away)
6. SfM recovers structure only up to a global scale factor

### What's Next

→ **Notebook 11**: 3D Representations — point clouds, meshes, voxels, implicit surfaces